## Imports

In [24]:
import pandas as pd
import joblib

from google.cloud import storage

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [31]:
PROJECT_ID = "project-0a6400db-0297-4323-a8f"

BUCKET_NAME = "mlops-course-project-0a6400db-0297-4323-a8f"
## for First Run
# TIMESTAMP = "2026-06-23_15-45-30" 

# print("Timestamp :", TIMESTAMP)

Timestamp : 2026-06-23_15-45-30


In [36]:
## for second run we have to change the timestamp
TIMESTAMP = "2026-06-23_15-26-49"
print("Timestamp :", TIMESTAMP)

Timestamp : 2026-06-23_15-26-49


In [37]:
!gcloud storage ls gs://$BUCKET_NAME/artifacts

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/


In [38]:
storage_client = storage.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)

In [39]:
MODEL_BLOB = f"artifacts/{TIMESTAMP}/model.joblib"
bucket.blob(MODEL_BLOB).download_to_filename("model.joblib")
print("Downloaded model.joblib")

Downloaded model.joblib


In [ ]:
EVAL_BLOB = "data/eval.csv"
bucket.blob(EVAL_BLOB).download_to_filename("eval.csv")
print("Downloaded eval.csv")

In [40]:
model = joblib.load("model.joblib")

eval_df = pd.read_csv("eval.csv")

print(eval_df.shape)

eval_df.head()

(30, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,4.4,3.0,1.3,0.2,setosa
1,6.1,3.0,4.9,1.8,virginica
2,4.9,2.4,3.3,1.0,versicolor
3,5.0,2.3,3.3,1.0,versicolor
4,4.4,3.2,1.3,0.2,setosa


In [41]:
X_eval = eval_df.drop("species", axis=1)
y_eval = eval_df["species"]
print("X_eval :", X_eval.shape)

print("y_eval :", y_eval.shape)

X_eval : (30, 4)
y_eval : (30,)


In [42]:
pred = model.predict(X_eval)

print("Predictions generated successfully!")

Predictions generated successfully!


In [43]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix)
accuracy = accuracy_score(y_eval, pred)
print("Accuracy :", accuracy)
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_eval, pred))
print("\nClassification Report:\n")

print(classification_report(y_eval, pred))

Accuracy : 0.9

Confusion Matrix:

[[10  0  0]
 [ 0  9  1]
 [ 0  2  8]]

Classification Report:

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [45]:
!gcloud storage ls gs://$BUCKET_NAME/artifacts

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/


In [46]:
!gcloud storage ls --recursive gs://$BUCKET_NAME/artifacts

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/:

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/:
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/eval.csv
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/logs.txt
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/metrics.json
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/model.joblib

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/:
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/eval.csv
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/logs.txt
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/metrics.json
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/model.joblib


## V1/data.csv for inference

In [47]:
# Download v1 data
!gcloud storage cp \
gs://$BUCKET_NAME/data/v1/data.csv \
v1.csv

# Load
v1_df = pd.read_csv("v1.csv")

# Separate features
X_v1 = v1_df.drop("species", axis=1)

# Predict
v1_pred = model.predict(X_v1)

# Save predictions
v1_df["prediction"] = v1_pred

v1_df.to_csv("predictions_v1.csv", index=False)

print(v1_df.head())

Copying gs://mlops-course-project-0a6400db-0297-4323-a8f/data/v1/data.csv to file://v1.csv
  Completed files 1/1 | 2.6kiB/2.6kiB                                          
   sepal_length  sepal_width  petal_length  petal_width species prediction
0           5.8          4.0           1.2          0.2  setosa     setosa
1           5.7          4.4           1.5          0.4  setosa     setosa
2           5.4          3.9           1.3          0.4  setosa     setosa
3           5.1          3.5           1.4          0.3  setosa     setosa
4           5.7          3.8           1.7          0.3  setosa     setosa


In [48]:
v1_df.to_csv(
    "predictions_v1.csv",
    index=False
)

print("predictions_v1.csv saved")

predictions_v1.csv saved


In [49]:
!gcloud storage cp \
predictions_v1.csv \
gs://$BUCKET_NAME/artifacts/$TIMESTAMP/predictions_v1.csv

print("predictions_v1.csv uploaded")

Copying file://predictions_v1.csv to gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/predictions_v1.csv
  Completed files 1/1 | 3.5kiB/3.5kiB                                          
predictions_v1.csv uploaded


## V2/data.csv for inference

In [53]:
# Download v2 data
!gcloud storage cp \
gs://$BUCKET_NAME/data/v2/data.csv \
v2.csv

# Load
v2_df = pd.read_csv("v1.csv")

# Separate features
X_v2 = v2_df.drop("species", axis=1)

# Predict
v2_pred = model.predict(X_v2)

# Save predictions
v2_df["prediction"] = v2_pred

v2_df.to_csv("predictions_v2.csv", index=False)

print(v2_df.head())

Copying gs://mlops-course-project-0a6400db-0297-4323-a8f/data/v2/data.csv to file://v2.csv
  Completed files 1/1 | 1.3kiB/1.3kiB                                          
   sepal_length  sepal_width  petal_length  petal_width species prediction
0           5.8          4.0           1.2          0.2  setosa     setosa
1           5.7          4.4           1.5          0.4  setosa     setosa
2           5.4          3.9           1.3          0.4  setosa     setosa
3           5.1          3.5           1.4          0.3  setosa     setosa
4           5.7          3.8           1.7          0.3  setosa     setosa


In [56]:
v2_df.to_csv(
    "predictions_v2.csv",
    index=False)

print("predictions_v2.csv saved")

predictions_v2.csv saved


In [57]:
!gcloud storage cp \
predictions_v2.csv \
gs://$BUCKET_NAME/artifacts/$TIMESTAMP/predictions_v2.csv

print("predictions_v2.csv uploaded")

Copying file://predictions_v2.csv to gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/predictions_v2.csv
  Completed files 1/1 | 3.5kiB/3.5kiB                                          
predictions_v2.csv uploaded


In [58]:
!gcloud storage ls --recursive gs://$BUCKET_NAME/artifacts

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/:

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/:
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/eval.csv
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/logs.txt
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/metrics.json
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-08-49/model.joblib

gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/:
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/eval.csv
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/logs.txt
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/metrics.json
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifacts/2026-06-23_15-26-49/model.joblib
gs://mlops-course-project-0a6400db-0297-4323-a8f/artifa

### Task 6 Comparision 

In [59]:
!gcloud storage cp \
gs://$BUCKET_NAME/data/v1/data.csv \
v1.csv

v1_df = pd.read_csv("v1.csv")

X_v1 = v1_df.drop("species", axis=1)

y_v1 = v1_df["species"]

pred_v1 = model.predict(X_v1)

from sklearn.metrics import accuracy_score

acc_v1 = accuracy_score(y_v1, pred_v1)

print("V1 Accuracy :", acc_v1)

Copying gs://mlops-course-project-0a6400db-0297-4323-a8f/data/v1/data.csv to file://v1.csv
  Completed files 1/1 | 2.6kiB/2.6kiB                                          
V1 Accuracy : 0.9702970297029703


In [60]:
!gcloud storage cp \
gs://$BUCKET_NAME/data/v2/data.csv \
v2.csv

v2_df = pd.read_csv("v2.csv")

X_v2 = v2_df.drop("species", axis=1)

y_v2 = v2_df["species"]

pred_v2 = model.predict(X_v2)

acc_v2 = accuracy_score(y_v2, pred_v2)

print("V2 Accuracy :", acc_v2)

Copying gs://mlops-course-project-0a6400db-0297-4323-a8f/data/v2/data.csv to file://v2.csv
  Completed files 1/1 | 1.3kiB/1.3kiB                                          
V2 Accuracy : 1.0


In [61]:
print("===== Comparison =====")

print("V1 Accuracy :", acc_v1)

print("V2 Accuracy :", acc_v2)

===== Comparison =====
V1 Accuracy : 0.9702970297029703
V2 Accuracy : 1.0
